# Minimal Token Sequence Test

Use this notebook to run tokens one by one, edit the sequence, and inspect the `State` after each step.

## Colab Setup

If you are running this from Colab, clone your repo first, then run the rest of the notebook from the repo root.

In [ ]:
# Uncomment and edit these lines in Colab if needed.
!git clone https://github.com/chahineNejm/graph_Time_series
%cd YOUR_REPO

## Load Current Token Files

This avoids importing the package-level `__init__`, so the notebook can run while the framework is still being assembled.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import types

import numpy as np


def find_repo_root():
    candidates = [Path.cwd()]
    candidates.extend(Path.cwd().glob("*"))
    for path in candidates:
        if (path / "graph_Time_series" / "state.py").exists():
            return path
    raise FileNotFoundError(
        "Could not find repo root. Expected graph_Time_series/state.py under "
        "the current folder or one of its direct children."
    )


ROOT = find_repo_root()
PACKAGE_DIR = ROOT / "graph_Time_series"
print("Repo root:", ROOT)


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pkg = types.ModuleType("graph_Time_series")
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault("graph_Time_series", pkg)

blocks = types.ModuleType("graph_Time_series.token_blocks")
blocks.__path__ = [str(PACKAGE_DIR / "token_blocks")]
sys.modules.setdefault("graph_Time_series.token_blocks", blocks)

state_mod = load_module("graph_Time_series.state", PACKAGE_DIR / "state.py")
token_mod = load_module("graph_Time_series.token", PACKAGE_DIR / "token.py")
norm_mod = load_module(
    "graph_Time_series.token_blocks.normalization",
    PACKAGE_DIR / "token_blocks" / "normalization.py",
)
rbf_mod = load_module(
    "graph_Time_series.token_blocks.kernel_rbf",
    PACKAGE_DIR / "token_blocks" / "kernel_rbf.py",
)

State = state_mod.State
ZNormalizationToken = norm_mod.ZNormalizationToken
KernelRBFToken = rbf_mod.KernelRBFToken

print("Loaded token test modules.")

## Choose Tokens

In [ ]:
TOKENS = {
    "ZNormalization": ZNormalizationToken(),
    "kernel_rbf": KernelRBFToken(),
}

# Edit this list to test your manual chain.
TOKEN_SEQUENCE = [
    "ZNormalization",
    "kernel_rbf",
]

## Load Electricity Data

This tries to load the long electricity config from GiftEval. A small holdout chunk is kept aside, and the token sequence runs on the working subset.

In [ ]:
USE_GIFTEVAL = True
DATASET_NAME = "Salesforce/GiftEvalParquet"
PREFERRED_CONFIGS = [
    "electricity_H_long",
]
MAX_RUN_SAMPLES = 24
HOLDOUT_SAMPLES = 8


def make_toy_data(n_samples=8, history_len=24, horizon=6):
    rng = np.random.default_rng(42)
    t_hist = np.linspace(0, 2 * np.pi, history_len)
    t_fut = np.linspace(2 * np.pi, 2.5 * np.pi, horizon)
    history = []
    future = []
    for i in range(n_samples):
        amplitude = 1.0 + 0.2 * i
        offset = 0.5 * i
        noise = 0.05 * rng.standard_normal(history_len)
        history.append(offset + amplitude * np.sin(t_hist) + noise)
        future.append(offset + amplitude * np.sin(t_fut))
    return np.asarray(history, dtype=np.float32), np.asarray(future, dtype=np.float32)


def load_gifteval_electricity(max_run_samples=24, holdout_samples=8):
    import subprocess
    import sys

    try:
        from datasets import get_dataset_config_names, load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import get_dataset_config_names, load_dataset

    configs = get_dataset_config_names(DATASET_NAME)
    electricity_configs = [c for c in configs if "electricity" in c.lower()]
    config = next((c for c in PREFERRED_CONFIGS if c in configs), None)
    if config is None:
        long_configs = [c for c in electricity_configs if "long" in c.lower()]
        config = long_configs[0] if long_configs else electricity_configs[0]

    print("Available electricity configs:", electricity_configs[:10])
    print("Using config:", config)
    ds = load_dataset(DATASET_NAME, config, split="test")

    histories = []
    futures = []
    needed = max_run_samples + holdout_samples
    for row in ds:
        ts = np.asarray(row["target"], dtype=np.float32)
        if ts.size == 0:
            continue
        if not np.all(np.isfinite(ts)):
            fill = np.nanmean(ts) if np.any(np.isfinite(ts)) else 0.0
            ts = np.nan_to_num(ts, nan=fill, posinf=fill, neginf=fill)
        horizon = row.get("prediction_length")
        horizon = int(horizon) if horizon is not None else max(1, len(ts) // 5)
        if len(ts) < horizon + 10:
            continue
        histories.append(ts[:-horizon])
        futures.append(ts[-horizon:])
        if len(histories) >= needed:
            break

    min_hist = min(len(x) for x in histories)
    min_fut = min(len(x) for x in futures)
    H_all = np.asarray([x[-min_hist:] for x in histories], dtype=np.float32)
    F_all = np.asarray([x[:min_fut] for x in futures], dtype=np.float32)

    run_n = min(max_run_samples, len(H_all))
    H = H_all[:run_n]
    F = F_all[:run_n]
    H_holdout = H_all[run_n:run_n + holdout_samples]
    F_holdout = F_all[run_n:run_n + holdout_samples]
    return H, F, H_holdout, F_holdout, config


if USE_GIFTEVAL:
    H, F, H_holdout, F_holdout, DATA_CONFIG = load_gifteval_electricity(
        MAX_RUN_SAMPLES, HOLDOUT_SAMPLES
    )
else:
    H, F = make_toy_data()
    H_holdout, F_holdout = make_toy_data(n_samples=2)
    DATA_CONFIG = "toy"

print("Run data:", H.shape, F.shape)
print("Held-out data:", H_holdout.shape, F_holdout.shape)

## Run Sequence

In [ ]:
def feature_shapes(values):
    return {k: getattr(v, "shape", None) for k, v in values.items()}


def summarize_state(state, label):
    print(f"\n--- {label} ---")
    print(state)
    print("token_sequence:", state.token_sequence)
    print("class_counts:", state.class_counts)
    print("historical_features:", feature_shapes(state.historical_features))
    print("future_features:", feature_shapes(state.future_features))
    print("flags:", state.flags)
    print("transforms:", [t.name for t in state.transform_stack])
    print("prediction_names:", state.prediction_names)
    print("active_target_base:", state.active_target_base.shape)
    print("current_target:", state.current_target.shape)


def run_sequence(sequence, H, F):
    state = State(H, F)
    summarize_state(state, "after init")

    for name in sequence:
        token = TOKENS[name]
        print(f"\nToken: {name}")
        can_apply = token.can_apply(state)
        print("can_apply:", can_apply)
        if not can_apply:
            raise RuntimeError(f"Token {name} cannot apply to current state")
        state = token.apply(state)
        summarize_state(state, f"after {name}")

    forecast = state.get_final_prediction()
    print("\nFinal forecast shape:", forecast.shape)
    print("Final forecast sample:", np.round(forecast[0], 3))
    return state, forecast


state, forecast = run_sequence(TOKEN_SEQUENCE, H, F)

## Inspect State

In [ ]:
print("token_sequence:", state.token_sequence)
print("class_counts:", state.class_counts)
print("historical_features:", list(state.historical_features.keys()))
print("future_features:", list(state.future_features.keys()))
print("flags:", state.flags)
print("transforms:", [t.name for t in state.transform_stack])
print("prediction_names:", state.prediction_names)

In [ ]:
state.print_log()